# AstroProcess: Pipeline Completo de Astrofotografía

Notebook interactiva para procesar imágenes astrónomicas desde archivos FITS crudos
hasta una imagen final en color.

### Modos soportados:

| Modo | Filtros | Asignación de color | Uso típico |
|------|---------|--------------------|-----------| 
| **SHO** (Paleta Hubble) | SII, Ha, OIII | S&rarr;R, H&rarr;G, O&rarr;B | Nebulosas de emisión |
| **RGB** (Color natural) | R, G, B | R&rarr;R, G&rarr;G, B&rarr;B | Galaxias, cúmulos |
| **HaRGB** | R, G, B + Ha | Ha se mezcla en luminancia roja | Galaxias con regiones HII |
| **HaOIII-RGB** | R, G, B + Ha + OIII | Ha&rarr;R, OIII&rarr;B mezclados con RGB | Objetos mixtos |

Todos los modos soportan **Luminancia (L)** opcional para agregar detalle.

### Características:

- Auto-detección de objetos y filtros disponibles
- Calibración completa (bias, dark, flat, pedestal)
- Alineación **interactiva con sliders** (resultado en vivo)
- Post-procesamiento **interactivo con sliders**
- Explicaciones en cada paso

---

## Paso 1: Instalar dependencias

- **OpenCV**: alineación y procesamiento de imagen
- **NumPy**: operaciones con arrays (las imágenes son arrays 2D)
- **SciPy**: aplicar desplazamientos sub-pixel
- **Matplotlib + ipympl**: visualización interactiva
- **ipywidgets**: sliders y controles interactivos
- **Pillow**: exportación PNG

In [ ]:
import subprocess, sys

for pkg in ['opencv-python', 'numpy', 'matplotlib', 'Pillow', 'scipy',
            'ipywidgets', 'ipympl']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("Dependencias instaladas.")

## Paso 2: Imports y funciones base

### Sobre los archivos FITS

**FITS** (Flexible Image Transport System) es el formato estándar en astronomía.
Cada archivo tiene una **cabecera** con metadatos (tamaño, tipo de dato, parámetros de captura)
y **datos** binarios (los píxeles) en formato big-endian.

Conceptos clave:

- **BITPIX**: tipo de dato (`16` = entero 16-bit, `-32` = float 32-bit)
- **BZERO/BSCALE**: transformación `valor_real = BZERO + BSCALE * raw`
- **OFFSET**: pedestal electrónico de la cámara (evita valores negativos por ruido)

In [ ]:
import numpy as np
import cv2
import re
from scipy.ndimage import shift as ndshift
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import os, glob
from pathlib import Path
from collections import defaultdict

# Intentar backend interactivo para matplotlib
# Si da error, reemplazar por: %matplotlib inline
%matplotlib widget

# === DIRECTORIOS ===
BASE_DIR = Path('.')
LIGHTS_DIR = BASE_DIR / 'lights'
BIAS_DIR   = BASE_DIR / 'biases'
DARKS_DIR  = BASE_DIR / 'darks'
FLATS_DIR  = BASE_DIR / 'flats'
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)


def read_fits(filepath):
    """Lee un archivo FITS. Retorna (header_dict, data_float32)."""
    with open(str(filepath), 'rb') as f:
        header = {}
        raw_hdr = b''
        end_found = False
        while not end_found:
            block = f.read(2880)
            if len(block) < 2880: break
            raw_hdr += block
            for i in range(0, 2880, 80):
                if block[i:i+3] == b'END': end_found = True; break
        data_start = len(raw_hdr)
        text = raw_hdr.decode('ascii', errors='ignore')
        for i in range(0, len(text), 80):
            card = text[i:i+80]; key = card[:8].strip()
            if key in ('END','COMMENT','HISTORY',''): continue
            if len(card) > 10 and card[8] == '=':
                vs = card[10:].strip()
                if vs.startswith("'"):
                    eq = vs.find("'", 1)
                    header[key] = vs[1:eq].strip() if eq > 0 else vs.strip("'").strip()
                    continue
                if '/' in vs: vs = vs[:vs.index('/')].strip()
                vs = vs.strip()
                if vs in ('T','F'): header[key] = (vs == 'T'); continue
                try: header[key] = float(vs) if ('.' in vs or 'E' in vs.upper()) else int(vs)
                except: header[key] = vs
        bitpix = int(header.get('BITPIX', -32))
        nx, ny = int(header.get('NAXIS1', 0)), int(header.get('NAXIS2', 0))
        bz = float(header.get('BZERO', 0.0))
        bs = float(header.get('BSCALE', 1.0))
        dt = np.dtype({16:'>i2', -32:'>f4', 32:'>i4', -64:'>f8'}[bitpix])
        f.seek(data_start)
        data = np.frombuffer(f.read(nx*ny*dt.itemsize), dtype=dt).reshape((ny, nx))
        data = np.float32(bz) + np.float32(bs) * data.astype(np.float32)
    return header, data


def write_fits(filepath, data):
    """Escribe float32 como FITS."""
    fp = str(filepath); ny, nx = data.shape
    cards = ["SIMPLE  =                    T / FITS standard",
             "BITPIX  =                  -32 / float32",
             "NAXIS   =                    2 / 2D image",
             f"NAXIS1  =             {nx:8d} / Width",
             f"NAXIS2  =             {ny:8d} / Height", "END"]
    hdr = b''.join(c.ljust(80).encode('ascii') for c in cards)
    hdr += b' ' * ((2880 - len(hdr) % 2880) % 2880)
    with open(fp, 'wb') as f:
        f.write(hdr); f.write(data.astype('>f4').tobytes())
    print(f"    Guardado: {os.path.basename(fp)}")

print("Funciones FITS listas.")

## Paso 3: Escaneo automático del directorio

La notebook escanea la carpeta `lights/` y detecta automáticamente:

- Qué **objetos** fueron fotografiados (ej. M97, NGC 2403)
- Qué **filtros** hay para cada objeto (H, O, S, R, G, B, L)
- Cuántos **frames** hay por filtro
- Qué **modo** conviene usar (SHO o RGB)

El nombre del archivo sigue el formato: `fecha_hora_FILTRO_OBJETO_exposicion_numero.fits`

In [ ]:
# Parsear nombres de archivo
# Formato: 2026-02-24_01-18-04_R_NGC 2403_240.00s_0000.fits
# El filtro es siempre una sola letra mayúscula en posición 3 del nombre

pattern = re.compile(
    r'^(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})_'  # fecha_hora
    r'([A-Z])_'                                     # filtro
    r'(.+?)_'                                       # objeto (puede tener espacios)
    r'(\d+\.\d+s)_'                                 # exposición
    r'(\d+)\.fits$'                                  # número
)

# Escanear lights
catalog = defaultdict(lambda: defaultdict(list))  # catalog[objeto][filtro] = [archivos]

for fn in sorted(os.listdir(str(LIGHTS_DIR))):
    if not fn.endswith('.fits'):
        continue
    m = pattern.match(fn)
    if m:
        timestamp, filt, obj, exp, num = m.groups()
        catalog[obj][filt].append({
            'file': str(LIGHTS_DIR / fn),
            'exp': float(exp.replace('s', '')),
            'name': fn
        })
    else:
        # Fallback: intentar detectar filtro de la posición habitual
        parts = fn.replace('.fits', '').split('_')
        if len(parts) >= 5:
            filt = parts[2]
            obj = parts[3]
            catalog[obj][filt].append({
                'file': str(LIGHTS_DIR / fn),
                'exp': 0,
                'name': fn
            })

# Mostrar resumen
print("=" * 60)
print("  OBJETOS DETECTADOS")
print("=" * 60)

for obj in sorted(catalog.keys()):
    filters = catalog[obj]
    has_sho = all(f in filters for f in ['H', 'O', 'S'])
    has_rgb = all(f in filters for f in ['R', 'G', 'B'])
    has_nb = any(f in filters for f in ['H', 'O', 'S'])
    has_L = 'L' in filters

    print(f"\n  {obj}:")
    for filt in sorted(filters.keys()):
        frames = filters[filt]
        exps = [f['exp'] for f in frames]
        print(f"    {filt}: {len(frames)} frames, exp={exps[0]:.0f}s")

    # Mostrar TODOS los modos posibles
    modes = []
    if has_sho:
        modes.append("SHO")
    if has_rgb:
        modes.append("RGB")
    if has_rgb and 'H' in filters:
        modes.append("HaRGB")
    if has_rgb and 'H' in filters and 'O' in filters:
        modes.append("HaOIII-RGB")
    if has_L and not has_rgb and not has_sho:
        modes.append("Luminancia (mono)")
    if not modes:
        modes.append("parcial")

    modes_str = ", ".join(modes)
    print(f"    Modos posibles: {modes_str}" + (" + L" if has_L and has_rgb or has_sho else ""))

print("\n" + "=" * 60)

## Paso 4: Seleccionar objeto y modo

Usa los **dropdowns** para seleccionar el objeto y el modo de composición.
Al cambiar de objeto, los modos se actualizan automáticamente.

| Modo | Filtros necesarios | Resultado | Ideal para |
|------|-------------------|-----------|-----------|
| **SHO** | Ha + OIII + SII | Falso color (Paleta Hubble) | Nebulosas de emisión |
| **RGB** | R + G + B | Color natural | Galaxias, cúmulos |
| **HaRGB** | R + G + B + Ha | Color natural + detalle Ha | Galaxias con regiones HII |
| **HaOIII-RGB** | R + G + B + Ha + OIII | RGB enriquecido con NB | Objetos mixtos |

### &iquest;Qué es HaRGB?

Cuando tienes **RGB + Ha**, podes mezclar el Ha en el canal rojo para resaltar
las regiones de hidrógeno ionizado (nebulosas, regiones HII en galaxias).
El resultado es color natural pero con nebulosidad mucho mas visible.

### &iquest;Qué es HaOIII-RGB?

Similar a HaRGB pero tambien mezcla OIII en el canal azul.
Resalta tanto regiones de hidrogeno (rojo) como de oxigeno (azul).

### Modo Luminancia (mono)

Si solo tenés frames de luminancia (L), la notebook genera una imagen
**monocromática** de alta calidad. Se calibra, alinea, apila y estira
como cualquier otro canal. El resultado es una imagen en escala de grises.

In [ ]:
# ==========================================================
# SELECCION INTERACTIVA - Usa los dropdowns para elegir
# ==========================================================

def get_modes_for_object(obj_name):
    """Calcula los modos posibles para un objeto dado."""
    filts = sorted(catalog[obj_name].keys())
    _has_sho = all(f in filts for f in ['H', 'O', 'S'])
    _has_rgb = all(f in filts for f in ['R', 'G', 'B'])
    _has_Ha = 'H' in filts
    _has_OIII = 'O' in filts
    _has_L = 'L' in filts
    modes = []
    if _has_sho: modes.append('SHO')
    if _has_rgb: modes.append('RGB')
    if _has_rgb and _has_Ha: modes.append('HaRGB')
    if _has_rgb and _has_Ha and _has_OIII: modes.append('HaOIII-RGB')
    if _has_L: modes.append('Lum')
    if not modes: modes.append('parcial')
    return modes

def auto_mode_for_object(obj_name):
    """Auto-detecta el mejor modo para un objeto."""
    filts = sorted(catalog[obj_name].keys())
    if all(f in filts for f in ['H', 'O', 'S']): return 'SHO'
    if all(f in filts for f in ['R', 'G', 'B']):
        if 'H' in filts and 'O' in filts: return 'HaOIII-RGB'
        if 'H' in filts: return 'HaRGB'
        return 'RGB'
    if 'L' in filts: return 'Lum'
    return 'parcial'

# --- Dropdown: Objeto ---
object_names = sorted(catalog.keys())
dd_object = widgets.Dropdown(
    options=object_names,
    value=object_names[0],
    description='Objeto:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

# --- Dropdown: Modo ---
initial_modes = get_modes_for_object(object_names[0])
dd_mode = widgets.Dropdown(
    options=initial_modes,
    value=auto_mode_for_object(object_names[0]),
    description='Modo:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

out_selection = widgets.Output()

def on_object_change(change):
    """Cuando cambia el objeto, actualizar modos disponibles."""
    obj = change['new']
    new_modes = get_modes_for_object(obj)
    dd_mode.options = new_modes
    dd_mode.value = auto_mode_for_object(obj)
    update_selection_info()

def update_selection_info():
    obj = dd_object.value
    mode = dd_mode.value
    filts = sorted(catalog[obj].keys())
    with out_selection:
        clear_output(wait=True)
        print(f"  Objeto:    {obj}")
        print(f"  Filtros:   {filts}")
        for f in sorted(catalog[obj].keys()):
            frames = catalog[obj][f]
            exps = set(fr['exp'] for fr in frames)
            exp_str = ', '.join(f'{e:.0f}s' for e in sorted(exps))
            print(f"    {f}: {len(frames)} frames, exp={exp_str}")
        print(f"  Modo:      {mode}")

dd_object.observe(on_object_change, names='value')
dd_mode.observe(lambda c: update_selection_info(), names='value')

display(widgets.VBox([
    widgets.HTML('<h3>Seleccion de objeto y modo</h3>'
                 '<p>1. Selecciona el objeto y el modo de composicion.<br>'
                 '2. Cuando estes conforme, <b>ejecuta la celda siguiente</b> para confirmar.</p>'),
    widgets.HBox([dd_object, widgets.HTML('&nbsp;&nbsp;'), dd_mode]),
]))
display(out_selection)
update_selection_info()

### Confirmar selección

Después de elegir el objeto y modo en los dropdowns de arriba,
**ejecuta esta celda** para confirmar la selección y continuar.

In [ ]:
# ==========================================================
# Lee la selección de los dropdowns
# ==========================================================
OBJECT = dd_object.value
MODE = dd_mode.value

# Detectar filtros disponibles
filters_available = sorted(catalog[OBJECT].keys())
has_sho = all(f in filters_available for f in ['H', 'O', 'S'])
has_rgb = all(f in filters_available for f in ['R', 'G', 'B'])
has_Ha = 'H' in filters_available
has_OIII = 'O' in filters_available
has_L = 'L' in filters_available

# Listar todos los modos posibles para este objeto
possible_modes = get_modes_for_object(OBJECT)

print(f"Objeto:          {OBJECT}")
print(f"Filtros:         {filters_available}")
print(f"Modos posibles:  {possible_modes}")
print(f"Modo elegido:    {MODE}")
print()

# Determinar filtros a procesar según el modo
if MODE == 'SHO':
    color_filters = ['H', 'O', 'S']
    nb_mix_filters = []
elif MODE == 'HaRGB':
    color_filters = ['R', 'G', 'B']
    nb_mix_filters = ['H']
elif MODE == 'HaOIII-RGB':
    color_filters = ['R', 'G', 'B']
    nb_mix_filters = ['H', 'O']
elif MODE == 'Lum':
    color_filters = ['L']
    nb_mix_filters = []
else:  # RGB
    color_filters = ['R', 'G', 'B']
    nb_mix_filters = []

# Agregar L como canal de luminancia extra (excepto en modo Lum donde ya es el principal)
extra_L = has_L and MODE != 'Lum' and 'L' not in color_filters
all_filters = list(dict.fromkeys(color_filters + nb_mix_filters + (['L'] if extra_L else [])))

print(f"Filtros color:       {color_filters}")
if nb_mix_filters:
    print(f"Filtros NB a mezclar: {nb_mix_filters}")
print(f"Total a procesar:    {all_filters}")

if MODE not in possible_modes and possible_modes:
    print(f"\nADVERTENCIA: modo '{MODE}' no es posible con estos filtros.")
    print(f"Modos validos: {possible_modes}")

## Paso 5: Diagnóstico de pedestal (OFFSET)

### &iquest;Que es el OFFSET?

La cámara agrega un **pedestal electrónico** a todos los pixeles para evitar
que el ruido de lectura produzca valores negativos. Si los másters de calibración
(bias, dark, flat) fueron tomados con un OFFSET diferente al de los lights,
hay que compensar la diferencia.

Medimos la mediana del bias (OFFSET alto) y de un light (OFFSET bajo)
para calcular la corrección necesaria.

In [ ]:
# Leer bias
_, bias_data = read_fits(BIAS_DIR / 'mbias.fits')
bias_med = np.median(bias_data)

# Leer un light de ejemplo
sample_file = catalog[OBJECT][color_filters[0]][0]['file']
_, sample_light = read_fits(sample_file)
light_med = np.median(sample_light)

PEDESTAL = bias_med - light_med

print(f"  Bias mediana:    {bias_med:.1f} ADU")
print(f"  Light mediana:   {light_med:.1f} ADU")
print(f"  Pedestal:        {PEDESTAL:.1f} ADU")

# Verificar darks disponibles
dark_files = sorted(glob.glob(str(DARKS_DIR / 'mdarks_*.fits')))
darks = {}
dark_exps = []
for df in dark_files:
    exp_str = os.path.basename(df).replace('mdarks_', '').replace('s.fits', '')
    try:
        exp = float(exp_str)
        _, darks[exp] = read_fits(df)
        dark_exps.append(exp)
        print(f"  Dark {exp:.0f}s: mediana={np.median(darks[exp]):.1f}")
    except: pass

# Mostrar matching de darks para cada filtro
print(f"\n  Darks disponibles: {sorted(dark_exps)}s")
print(f"  Verificando coincidencia con lights:")
for filt in all_filters:
    if filt not in catalog[OBJECT]: continue
    light_exp = catalog[OBJECT][filt][0]['exp']
    if light_exp == 0: continue
    best_exp = min(dark_exps, key=lambda d: abs(d - light_exp)) if dark_exps else 0
    diff = abs(best_exp - light_exp)
    if diff == 0:
        status = "EXACTO"
    elif diff <= 30:
        status = f"cercano (diff={diff:.0f}s) -> se escalara"
    else:
        status = f"MUY DIFERENTE (diff={diff:.0f}s) -> se escalara proporcionalmente"
    n_frames = len(catalog[OBJECT][filt])
    hot_warn = " [1 frame: hot pixels se corregiran con filtro mediano]" if n_frames == 1 else ""
    print(f"    {filt} ({light_exp:.0f}s, {n_frames}fr) -> dark {best_exp:.0f}s: {status}{hot_warn}")

# Flats disponibles
flats = {}
for filt in all_filters:
    fp = FLATS_DIR / f'mflat_{filt}.fits'
    if fp.exists():
        _, flats[filt] = read_fits(fp)
print(f"\n  Flats: {list(flats.keys())}")

## Paso 6: Calibración

### Fórmula:

```
calibrada = (light - dark_escalado) / flat
```

| Término | Corrige | Se obtiene de |
|---------|---------|---------------|
| **Dark** | Ruido térmico + patrón fijo | Foto con obturador cerrado, misma exposición |
| **Pedestal** | Diferencia de OFFSET entre másters y lights | `mediana(bias) - mediana(light)` |
| **Flat** | Viñeteo + polvo en sensor | Foto de campo uniforme, normalizada (media=1) |

### Escalado de darks

La corriente oscura (dark current) escala **linealmente** con el tiempo de exposición.
Si no tenemos un dark de exáctamente la misma exposición que el light, extraemos
la componente térmica pura `(dark - bias)` y la escalamos proporcionalmente:

```
dark_escalado = bias + (dark - bias) * (exp_light / exp_dark) - pedestal
```

Esto es crítico: un dark de 300s usado sin escalar en un light de 400s deja ~25%
de los hot pixels sin corregir.

### Corrección cosmética (hot pixels)

Los **hot pixels** son pixeles defectuosos que acumulan carga extra.
Con 3+ frames, el apilado por mediana los elimina automáticamente.
Pero con **1 solo frame** (como Ha u OIII a veces), necesitamos detectarlos
comparando cada pixel con sus vecinos y reemplazar los outliers.

In [ ]:
from scipy.ndimage import median_filter

def scale_dark(dark, bias, light_exp, dark_exp, pedestal):
    """Escala un dark para que coincida con la exposición del light.

    La corriente oscura es lineal con el tiempo:
      dark_escalado = bias + (dark - bias) * (light_exp / dark_exp) - pedestal

    Así extraemos solo la parte térmica, la escalamos, y compensamos el pedestal.
    """
    thermal = dark - bias  # componente térmica pura
    scale = light_exp / dark_exp if dark_exp > 0 else 1.0
    return bias + thermal * scale - pedestal

def cosmetic_correction(data, sigma=5.0, size=5):
    """Detecta y corrige hot/cold pixels comparando con mediana local.

    Un píxel se considera 'hot' si difiere de la mediana de sus vecinos
    por mas de sigma * desviacion_estandar_local. Se reemplaza por la mediana.
    Esencial para frames individuales donde no hay apilado que los elimine.
    """
    med = median_filter(data, size=size)
    diff = np.abs(data - med)
    # Estimación robusta del ruido local
    mad = median_filter(diff, size=size * 2 + 1)
    threshold = sigma * np.clip(mad, 1.0, None)  # evitar division por 0
    mask = diff > threshold
    corrected = data.copy()
    corrected[mask] = med[mask]
    n_fixed = np.sum(mask)
    return corrected, n_fixed

def calibrate(light, dark_scaled, flat):
    """Calibra un frame: (light - dark_escalado) / flat"""
    return (light - dark_scaled) / np.clip(flat, 0.3, 3.0)

def get_best_dark(exp, darks_dict):
    """Selecciona el dark con exposicion mas cercana."""
    if not darks_dict:
        return None, 0
    best_exp = min(darks_dict.keys(), key=lambda d: abs(d - exp))
    return darks_dict[best_exp], best_exp

print(f"Calibrando {OBJECT} ({MODE})...\n")
calibrated = defaultdict(list)      # calibrated[filt] = [array, ...]
calibrated_exps = defaultdict(list)  # calibrated_exps[filt] = [exp_seconds, ...]

for filt in all_filters:
    if filt not in catalog[OBJECT]:
        print(f"  {filt}: no hay frames, saltando")
        continue
    flat = flats.get(filt, np.ones_like(bias_data))
    n_frames = len(catalog[OBJECT][filt])
    for frame_info in catalog[OBJECT][filt]:
        fp = frame_info['file']
        exp = frame_info['exp']
        hdr, light = read_fits(fp)
        # Si exp es 0 (no se pudo parsear), intentar del header
        if exp == 0:
            exp = float(hdr.get('EXPTIME', 300))
        dark_raw, dark_exp = get_best_dark(exp, darks)
        if dark_raw is None:
            print(f"  ADVERTENCIA: no hay dark, saltando {os.path.basename(fp)}")
            continue
        # Escalar dark a la exposición correcta
        dark_scaled = scale_dark(dark_raw, bias_data, exp, dark_exp, PEDESTAL)
        if abs(dark_exp - exp) > 1:
            scale_info = f" (dark {dark_exp:.0f}s escalado a {exp:.0f}s)"
        else:
            scale_info = ""
        cal = calibrate(light, dark_scaled, flat)
        # Corrección cosmética (hot pixels) - especialmente crítico para frames individuales
        cal, n_hot = cosmetic_correction(cal, sigma=5.0, size=5)
        calibrated[filt].append(cal)
        calibrated_exps[filt].append(exp)
        hot_info = f", {n_hot} hot px corregidos" if n_hot > 0 else ""
        print(f"  {filt} {os.path.basename(fp)}: med={np.median(cal):.1f}{scale_info}{hot_info}")

print("\nCalibración completada.")
print("Filtros calibrados:", {f: len(calibrated[f]) for f in calibrated})

## Paso 7: Alineación intra-filtro + Apilado

### Alineación intra-filtro

Antes de apilar, alineamos los frames **dentro** de cada filtro.
Entre toma y toma el telescopio se mueve ligeramente.
Usamos **correlación de fase** (FFT) para detectar el desplazamiento.

### Apilado por mediana

La **mediana** es robusta contra outliers (rayos cósmicos, satelites, pixeles calientes).
Con 3 frames: si un pixel tiene valores [100, 102, 50000], la mediana es 102.

### Exposiciones mixtas

Si un filtro tiene frames con diferentes tiempos de exposición (ej: 30s, 60s, 120s),
la notebook los **normaliza a ADU/segundo** antes de alinear y apilar.
Esto evita que la correlación de fase falle por diferencias de brillo.

> Si ya existen stacks en `output/`, se reutilizan. Pon `FORCE_RESTACK = True` para regenerar.

In [ ]:
FORCE_RESTACK = False  # True para re-apilar aunque ya existan

def phase_align(ref, target, label=""):
    """Alinea target a ref por correlacion de fase."""
    h, w = ref.shape
    def norm(img):
        p1, p99 = np.percentile(img, [1, 99.5])
        return np.clip((img - p1) / max(p99 - p1, 1), 0, 1).astype(np.float64)
    ref_n, tgt_n = norm(ref), norm(target)
    # Reducir para velocidad
    s = 0.125
    r_s = cv2.resize(ref_n, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    t_s = cv2.resize(tgt_n, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    hs, ws = r_s.shape
    hann = cv2.createHanningWindow((ws, hs), cv2.CV_64F)
    (dx_s, dy_s), conf = cv2.phaseCorrelate(r_s * hann, t_s * hann)
    dx, dy = dx_s / s, dy_s / s
    if abs(dx) > 500 or abs(dy) > 500 or conf < 0.005:
        if label: print(f"    {label}: shift descartado (dx={dx:.0f}, dy={dy:.0f}, conf={conf:.3f})")
        return target
    if label: print(f"    {label}: dx={dx:+.1f}, dy={dy:+.1f}, conf={conf:.3f}")
    return ndshift(target, [dy, dx], order=1, mode='constant', cval=float(np.median(target)))

# Prefijo para stacks (incluye objeto para no mezclar)
obj_tag = OBJECT.replace(' ', '_')

# Alinear + apilar
print(f"Alineando y apilando {OBJECT}...\n")
stacked = {}

for filt in all_filters:
    stack_file = OUTPUT_DIR / f'stacked_{obj_tag}_{filt}.fits'

    if stack_file.exists() and not FORCE_RESTACK:
        _, stacked[filt] = read_fits(stack_file)
        print(f"  {filt}: REUTILIZADO ({stack_file.name})")
        continue

    frames = calibrated.get(filt, [])
    exps = calibrated_exps.get(filt, [])
    if not frames:
        continue

    # Si hay diferentes tiempos de exposición, normalizar a ADU/segundo
    # antes de alinear y apilar. Esto es crítico para objetos como cúmulos
    # donde se toman frames de diferente duración.
    unique_exps = set(exps)
    if len(unique_exps) > 1 and all(e > 0 for e in exps):
        print(f"  {filt}: exposiciones mixtas ({sorted(unique_exps)}s) -> normalizando a ADU/s")
        frames = [f / e for f, e in zip(frames, exps)]

    # Alinear intra-filtro
    if len(frames) > 1:
        print(f"  {filt}: alineando {len(frames)} frames...")
        aligned_f = [frames[0]]
        for i in range(1, len(frames)):
            aligned_f.append(phase_align(frames[0], frames[i], f"frame {i}"))
        stack = np.median(np.array(aligned_f), axis=0)
    else:
        stack = frames[0]

    stacked[filt] = stack
    write_fits(str(stack_file), stack)
    print(f"  {filt}: apilado (med={np.median(stack):.1f}, P99.9={np.percentile(stack, 99.9):.0f})")

# Visualizar
n = len(stacked)
fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4))
if n == 1: axes = [axes]
for ax, (f, d) in zip(axes, stacked.items()):
    v1, v2 = np.percentile(d, [1, 99.5])
    ax.imshow(d, cmap='gray', vmin=v1, vmax=v2, origin='lower')
    ax.set_title(f'Stacked {f}', fontsize=11); ax.axis('off')
plt.suptitle(f'{OBJECT} - Canales apilados', fontsize=13)
plt.tight_layout(); plt.show()

print(f"\nStacks listos: {list(stacked.keys())}")

## Paso 8: Alineación inter-filtro (interactiva)

Los canales de diferentes filtros pueden estar desplazados entre sí
(cambio de filtro, flexión diferencial).

### &iquest;Cómo usar los sliders?

1. Mira el **zoom** (derecha): las estrellas deben verse **blancas y redondas**
2. Si ves franjas de color (rojo/verde/azul separados), ajusta los sliders `dx` y `dy`
3. Usa los controles de **Zoom X/Y** para navegar a diferentes estrellas
4. Cuando estes conforme, ejecuta la **celda siguiente** para aplicar a resolución completa

Los valores iniciales vienen de la auto-detección. A veces solo necesitan ajustes finos.

**Importante**: En modos HaRGB y HaOIII-RGB, el preview muestra la mezcla real
de narrowband, asi que podes ver si Ha u OIII estan desalineados respecto al RGB.

In [ ]:
# Preparar previews (1/8 escala para velocidad)
PREVIEW_SCALE = 0.125

def quick_stretch(data):
    p1, p999 = np.percentile(data, [1, 99.9])
    d = np.clip((data - p1) / max(p999 - p1, 1), 0, 1)
    return np.power(d, 0.3)

previews = {}
for filt in stacked:
    small = cv2.resize(stacked[filt].astype(np.float32), None,
                       fx=PREVIEW_SCALE, fy=PREVIEW_SCALE, interpolation=cv2.INTER_AREA)
    previews[filt] = quick_stretch(small)

h_prev, w_prev = list(previews.values())[0].shape
ref_filt = color_filters[0]  # H para SHO, R para RGB

# Auto-detectar shifts iniciales
auto_shifts = {}
for filt in all_filters:
    if filt == ref_filt or filt not in stacked:
        continue
    ref_s = cv2.resize(stacked[ref_filt].astype(np.float32), None,
                       fx=PREVIEW_SCALE, fy=PREVIEW_SCALE, interpolation=cv2.INTER_AREA)
    tgt_s = cv2.resize(stacked[filt].astype(np.float32), None,
                       fx=PREVIEW_SCALE, fy=PREVIEW_SCALE, interpolation=cv2.INTER_AREA)
    def norm64(img):
        p1, p99 = np.percentile(img, [1, 99.5])
        return np.clip((img - p1) / max(p99 - p1, 1), 0, 1).astype(np.float64)
    r_n, t_n = norm64(ref_s), norm64(tgt_s)
    hs, ws = r_n.shape
    hann = cv2.createHanningWindow((ws, hs), cv2.CV_64F)
    (dx_s, dy_s), conf = cv2.phaseCorrelate(r_n * hann, t_n * hann)
    dx, dy = dx_s / PREVIEW_SCALE, dy_s / PREVIEW_SCALE
    if abs(dx) < 100 and abs(dy) < 100 and conf > 0.01:
        auto_shifts[filt] = (round(dx, 1), round(dy, 1))
    else:
        auto_shifts[filt] = (0.0, 0.0)
    print(f"  {filt}->{ref_filt}: dx={dx:+.1f}, dy={dy:+.1f} (conf={conf:.3f})"
          f"{' [auto]' if filt in auto_shifts and auto_shifts[filt] != (0,0) else ''}")

# Crear sliders
sliders = {}
filters_to_align = [f for f in all_filters if f != ref_filt and f in stacked]

for filt in filters_to_align:
    init_dx, init_dy = auto_shifts.get(filt, (0, 0))
    sliders[f'{filt}_dx'] = widgets.FloatSlider(
        value=init_dx, min=-60, max=60, step=0.5,
        description=f'{filt} dx:', continuous_update=True,
        style={'description_width': '50px'}, layout=widgets.Layout(width='380px'))
    sliders[f'{filt}_dy'] = widgets.FloatSlider(
        value=init_dy, min=-60, max=60, step=0.5,
        description=f'{filt} dy:', continuous_update=True,
        style={'description_width': '50px'}, layout=widgets.Layout(width='380px'))

zoom_x = widgets.IntSlider(value=w_prev//2, min=80, max=w_prev-80, step=5,
    description='Zoom X:', style={'description_width': '60px'},
    layout=widgets.Layout(width='380px'))
zoom_y = widgets.IntSlider(value=h_prev//2, min=80, max=h_prev-80, step=5,
    description='Zoom Y:', style={'description_width': '60px'},
    layout=widgets.Layout(width='380px'))
zoom_size = widgets.IntSlider(value=80, min=30, max=200, step=10,
    description='Tamano:', style={'description_width': '60px'},
    layout=widgets.Layout(width='380px'))

out_align = widgets.Output()

def update_alignment(**kwargs):
    with out_align:
        clear_output(wait=True)
        shifted = {}
        shifted[ref_filt] = previews[ref_filt]
        for filt in filters_to_align:
            dx = sliders.get(f'{filt}_dx')
            dy = sliders.get(f'{filt}_dy')
            if dx is None:
                shifted[filt] = previews.get(filt, np.zeros_like(previews[ref_filt]))
                continue
            dx_px = dx.value * PREVIEW_SCALE
            dy_px = dy.value * PREVIEW_SCALE
            if abs(dx_px) < 0.05 and abs(dy_px) < 0.05:
                shifted[filt] = previews[filt]
            else:
                shifted[filt] = ndshift(previews[filt], [dy_px, dx_px],
                                        order=1, mode='constant', cval=0)
        # Componer RGB según modo (incluyendo mezcla NB para ver efecto real)
        zeros = np.zeros((h_prev, w_prev))
        if MODE == 'SHO':
            r_ch = shifted.get('S', zeros)
            g_ch = shifted.get('H', zeros)
            b_ch = shifted.get('O', zeros)
        elif MODE == 'HaRGB':
            r_rgb = shifted.get('R', zeros)
            ha_ch = shifted.get('H', zeros)
            r_ch = np.clip(0.65 * r_rgb + 0.35 * ha_ch, 0, 1)
            g_ch = shifted.get('G', zeros)
            b_ch = shifted.get('B', zeros)
        elif MODE == 'HaOIII-RGB':
            r_rgb = shifted.get('R', zeros)
            ha_ch = shifted.get('H', zeros)
            b_rgb = shifted.get('B', zeros)
            oiii_ch = shifted.get('O', zeros)
            r_ch = np.clip(0.65 * r_rgb + 0.35 * ha_ch, 0, 1)
            g_ch = shifted.get('G', zeros)
            b_ch = np.clip(0.65 * b_rgb + 0.35 * oiii_ch, 0, 1)
        else:
            r_ch = shifted.get('R', zeros)
            g_ch = shifted.get('G', zeros)
            b_ch = shifted.get('B', zeros)
        rgb = np.clip(np.stack([r_ch, g_ch, b_ch], axis=-1), 0, 1).astype(np.float32)
        cx, cy, zs = zoom_x.value, zoom_y.value, zoom_size.value
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        axes[0].imshow(rgb, origin='lower')
        rect = plt.Rectangle((cx-zs, cy-zs), 2*zs, 2*zs, lw=2, ec='yellow', fc='none')
        axes[0].add_patch(rect)
        axes[0].set_title(f'{OBJECT} - {MODE}', fontsize=11); axes[0].axis('off')
        y1, y2 = max(0, cy-zs), min(h_prev, cy+zs)
        x1, x2 = max(0, cx-zs), min(w_prev, cx+zs)
        axes[1].imshow(rgb[y1:y2, x1:x2], origin='lower', interpolation='nearest')
        axes[1].set_title('Zoom (estrellas = blancas, sin franjas)', fontsize=11); axes[1].axis('off')
        plt.tight_layout(); plt.show()

for ctrl in list(sliders.values()) + [zoom_x, zoom_y, zoom_size]:
    ctrl.observe(lambda c: update_alignment(), names='value')

filter_boxes = []
for filt in filters_to_align:
    if f'{filt}_dx' in sliders:
        name_map = {'O':'OIII','S':'SII','H':'Ha','R':'Rojo','G':'Verde','B':'Azul','L':'Lum'}
        filter_boxes.append(widgets.VBox([
            widgets.HTML(f'<b>{filt} ({name_map.get(filt, filt)})</b>'),
            sliders[f'{filt}_dx'], sliders[f'{filt}_dy']]))

display(widgets.VBox([
    widgets.HTML(f'<h3>Alineacion inter-filtro: {MODE} ({OBJECT})</h3>'
                 f'<p>Referencia: {ref_filt}. Mover sliders hasta que estrellas sean blancas.</p>'),
    widgets.HBox(filter_boxes),
    widgets.HTML('<br>'),
    widgets.VBox([widgets.HTML('<b>Navegacion del zoom</b>'), zoom_x, zoom_y, zoom_size]),
]))
display(out_align)
update_alignment()

## Paso 9: Aplicar alineación a resolución completa

Una vez conforme con los sliders, ejecuta esta celda.
Aplica los mismos desplazamientos a las imágenes de resolución completa.

In [ ]:
print("Aplicando alineación a resolución completa...\n")
aligned = {ref_filt: stacked[ref_filt]}

for filt in filters_to_align:
    if filt not in stacked:
        continue
    dx_s = sliders.get(f'{filt}_dx')
    dy_s = sliders.get(f'{filt}_dy')
    dx = dx_s.value if dx_s else 0
    dy = dy_s.value if dy_s else 0
    if abs(dx) < 0.1 and abs(dy) < 0.1:
        aligned[filt] = stacked[filt]
        print(f"  {filt}: sin shift")
    else:
        print(f"  {filt}: ({dy:+.1f}, {dx:+.1f})...", end=" ", flush=True)
        aligned[filt] = ndshift(stacked[filt], [dy, dx], order=1,
                                mode='constant', cval=float(np.median(stacked[filt])))
        print("OK")

print("\nAlineación aplicada.")

## Paso 10: Composición de color

### Modos de composición:

- **SHO**: SII&rarr;R, Ha&rarr;G, OIII&rarr;B (falso color, revela química)
- **RGB**: R&rarr;R, G&rarr;G, B&rarr;B (color natural)
- **HaRGB**: RGB + Ha mezclado en canal rojo (resalta regiones HII)
- **HaOIII-RGB**: RGB + Ha en rojo + OIII en azul (resalta emisión)

### &iquest;Cómo funciona la mezcla NB en HaRGB / HaOIII-RGB?

El parámetro `NB_MIX` controla cuánto narrowband se mezcla:

```
R_final = (1 - NB_MIX) * R  +  NB_MIX * Ha
B_final = (1 - NB_MIX) * B  +  NB_MIX * OIII   (solo HaOIII-RGB)
```

| NB_MIX | Efecto |
|--------|--------|
| 0.0 | RGB puro (ignora narrowband) |
| 0.3 | Mezcla suave (recomendado) |
| 0.5 | Mezcla media |
| 0.7+ | NB dominante (puede verse artificial) |

### Stretch por modo:

| Parámetro | SHO | RGB/HaRGB | Efecto |
|-----------|-----|-----------|--------|
| `STRETCH` | 0.05-0.1 | 0.2-0.5 | Menor = más agresivo |
| `GAMMA` | 3.0-4.0 | 2.0-3.0 | Mayor = más brillo medios |
| `WHITE_PCT` | 99.95 | 99.9 | Percentil punto blanco |

In [ ]:
def prepare_channel(data, white_pct=99.95):
    clean = np.clip(data, 0, None)
    white = np.percentile(clean, white_pct)
    if white <= 0: return np.zeros_like(data)
    return np.clip(clean / white, 0, 1)

def asinh_stretch(data, strength=0.08):
    factor = 1.0 / max(strength, 0.001)
    return np.arcsinh(data * factor) / np.arcsinh(factor)

def gamma_corr(data, gamma=3.5):
    return np.power(np.clip(data, 0, 1), 1.0 / gamma)

# =============================================
# Defaults según modo
# =============================================
if MODE == 'SHO':
    STRETCH   = 0.08
    GAMMA     = 3.5
    WHITE_PCT = 99.95
    NB_MIX    = 0     # No aplica en SHO
else:
    STRETCH   = 0.3
    GAMMA     = 2.5
    WHITE_PCT = 99.9
    NB_MIX    = 0.35  # Cuánto NB mezclar en HaRGB/HaOIII-RGB
# =============================================

print(f"Modo: {MODE}")
print(f"STRETCH={STRETCH}, GAMMA={GAMMA}, WHITE_PCT={WHITE_PCT}")
if MODE in ('HaRGB', 'HaOIII-RGB'):
    print(f"NB_MIX={NB_MIX} (mezcla de narrowband en RGB)")
print()

def process_ch(data):
    return gamma_corr(asinh_stretch(prepare_channel(data, WHITE_PCT), STRETCH), GAMMA)

# --- Composición según modo ---
if MODE == 'SHO':
    r = process_ch(aligned['S'])
    g = process_ch(aligned['H'])
    b = process_ch(aligned['O'])
    labels = ['SII (Rojo)', 'Ha (Verde)', 'OIII (Azul)']

elif MODE == 'HaRGB':
    r_rgb = process_ch(aligned['R'])
    g = process_ch(aligned['G'])
    b = process_ch(aligned['B'])
    ha = process_ch(aligned['H'])
    # Mezclar Ha en canal rojo
    r = np.clip((1 - NB_MIX) * r_rgb + NB_MIX * ha, 0, 1)
    labels = [f'R + Ha (mix={NB_MIX})', 'G (Verde)', 'B (Azul)']
    print(f"  Ha mezclado en Rojo con peso {NB_MIX}")

elif MODE == 'HaOIII-RGB':
    r_rgb = process_ch(aligned['R'])
    g = process_ch(aligned['G'])
    b_rgb = process_ch(aligned['B'])
    ha = process_ch(aligned['H'])
    oiii = process_ch(aligned['O'])
    # Mezclar Ha en rojo, OIII en azul
    r = np.clip((1 - NB_MIX) * r_rgb + NB_MIX * ha, 0, 1)
    b = np.clip((1 - NB_MIX) * b_rgb + NB_MIX * oiii, 0, 1)
    labels = [f'R + Ha (mix={NB_MIX})', 'G (Verde)', f'B + OIII (mix={NB_MIX})']
    print(f"  Ha mezclado en Rojo, OIII mezclado en Azul (peso {NB_MIX})")

elif MODE == 'Lum':
    # Modo monocromático: sólo luminancia
    lum = process_ch(aligned['L'])
    r, g, b = lum, lum, lum
    labels = ['Luminancia', '', '']
    print("  Modo Luminancia: imagen monocromática")

else:  # RGB
    r = process_ch(aligned['R'])
    g = process_ch(aligned['G'])
    b = process_ch(aligned['B'])
    labels = ['R (Rojo)', 'G (Verde)', 'B (Azul)']

composed = np.clip(np.stack([r, g, b], axis=-1), 0, 1).astype(np.float32)

if MODE == 'Lum':
    # Para modo Lum, mostrar sólo la imagen en escala de grises
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(lum, cmap='gray', vmin=0, vmax=1, origin='lower')
    ax.set_title(f'{OBJECT} - Luminancia', fontsize=14); ax.axis('off')
    plt.tight_layout(); plt.show()
    print(f"  Luminancia: media={lum.mean():.3f}, mediana={np.median(lum):.3f}")
else:
    for name, ch in zip(labels, [r, g, b]):
        if name:
            print(f"  {name}: media={ch.mean():.3f}, mediana={np.median(ch):.3f}")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    cmaps = ['Reds', 'Greens', 'Blues']
    for ax, (name, ch, cm) in zip(axes.flat[:3], zip(labels, [r, g, b], cmaps)):
        ax.imshow(ch, cmap=cm, vmin=0, vmax=1, origin='lower')
        ax.set_title(name, fontsize=12); ax.axis('off')
    axes[1,1].imshow(composed, origin='lower')
    axes[1,1].set_title(f'{MODE} Combinado', fontsize=12); axes[1,1].axis('off')
    plt.suptitle(f'{OBJECT} - {MODE}', fontsize=14)
    plt.tight_layout(); plt.show()

## Paso 11: Luminancia (LRGB) - Opcional

La luminancia agrega **detalle y nitidez** (especialmente en estrellas)
sin cambiar los colores.

| LUM_WEIGHT | Efecto |
|------------|--------|
| 0.0 | Sin luminancia (solo color puro) |
| 0.2-0.3 | Mezcla suave (agrega detalle, conserva color) |
| 0.5 | Mezcla media |
| 1.0 | Sólo luminancia (pierde color, NO recomendado) |

Con pocos frames L (1-2), usar 0.0 o 0.2 máximo.

In [ ]:
LUM_WEIGHT = 0.3  # Ajustar segun cantidad de datos L

if MODE == 'Lum':
    final_pre = composed.copy()
    print("Modo Luminancia: LRGB no aplica.")
elif 'L' in aligned and LUM_WEIGHT > 0:
    l_ch = process_ch(aligned['L'])
    sho_8 = (composed * 255).astype(np.uint8)
    hsv = cv2.cvtColor(sho_8, cv2.COLOR_RGB2HSV).astype(np.float32)
    v_sho = hsv[:,:,2] / 255.0
    hsv[:,:,2] = np.clip((LUM_WEIGHT * l_ch + (1 - LUM_WEIGHT) * v_sho) * 255, 0, 255)
    final_pre = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32) / 255.0
    print(f"LRGB aplicado (peso={LUM_WEIGHT})")
else:
    final_pre = composed.copy()
    print(f"Sin luminancia (LUM_WEIGHT={LUM_WEIGHT})")

## Paso 12: Post-procesamiento interactivo

Usa los sliders para ajustar color, saturación y brillo **en tiempo real**.

### Guía de parámetros:

| Slider | Qué hace | Valores típicos |
|--------|----------|----------------|
| **R / G / B** | Balance de color por canal | 0.5 - 1.5 |
| **Saturación** | Intensidad del color | 1.0 - 2.5 (SHO: >1.5, RGB: ~1.2) |
| **SCNR** | Reduce exceso de verde (SHO) | 0.5=agresivo, 1.0+=desactivado |
| **Gamma extra** | Aclara tonos medios | 1.0=nada, 1.5=bastante |
| **Denoise** | Suaviza ruido | 0=off, 5=normal, 9=fuerte |

### Errores comunes:

- **Saturación < 1.0**: REDUCE color (todo gris). Siempre &ge; 1.0
- **SCNR muy bajo**: quita todo el verde &rarr; imagen purpura
- **Gamma muy alto**: imagen lavada, pierde contraste

In [ ]:
# Preparar preview para post-procesado
preview_pre = cv2.resize(final_pre, None, fx=PREVIEW_SCALE, fy=PREVIEW_SCALE,
                         interpolation=cv2.INTER_AREA)

# Defaults según modo
if MODE == 'SHO':
    def_R, def_G, def_B = 1.0, 0.75, 1.1
    def_sat = 1.8
    def_scnr = 0.85
else:
    def_R, def_G, def_B = 1.0, 1.0, 1.0
    def_sat = 1.2
    def_scnr = 1.1  # desactivado para RGB

# Sliders de post-procesado
pp = {}
pp['R']       = widgets.FloatSlider(value=def_R, min=0.3, max=1.8, step=0.05,
                    description='Rojo:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))
pp['G']       = widgets.FloatSlider(value=def_G, min=0.3, max=1.8, step=0.05,
                    description='Verde:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))
pp['B']       = widgets.FloatSlider(value=def_B, min=0.3, max=1.8, step=0.05,
                    description='Azul:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))
pp['sat']     = widgets.FloatSlider(value=def_sat, min=0.5, max=3.0, step=0.1,
                    description='Saturacion:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))
pp['scnr']    = widgets.FloatSlider(value=def_scnr, min=0.3, max=1.5, step=0.05,
                    description='SCNR:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))
pp['gamma']   = widgets.FloatSlider(value=1.2, min=0.5, max=2.5, step=0.1,
                    description='Gamma extra:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))
pp['denoise'] = widgets.IntSlider(value=5, min=0, max=11, step=2,
                    description='Denoise:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))

out_pp = widgets.Output()

def apply_postprocess(img, R, G, B, sat, scnr, gamma, denoise):
    """Aplica post-procesado a una imagen RGB [0,1]."""
    proc = img.copy()
    # Color balance
    proc[:,:,0] *= R
    proc[:,:,1] *= G
    proc[:,:,2] *= B
    proc = np.clip(proc, 0, 1)
    # SCNR
    if scnr <= 1.0:
        r_c, g_c, b_c = proc[:,:,0], proc[:,:,1], proc[:,:,2]
        limit = (r_c + b_c) / 2.0 * scnr + g_c * (1 - scnr)
        proc[:,:,1] = np.minimum(g_c, limit)
        proc = np.clip(proc, 0, 1)
    # Saturacion
    p8 = (proc * 255).astype(np.uint8)
    hsv = cv2.cvtColor(p8, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:,:,1] = np.clip(hsv[:,:,1] * sat, 0, 255)
    proc = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32) / 255.0
    # Gamma
    if gamma != 1.0:
        proc = np.power(np.clip(proc, 0, 1), 1.0 / gamma)
    # Fondo
    for ch in range(3):
        bg = np.percentile(proc[:,:,ch], 15)
        proc[:,:,ch] = np.clip(proc[:,:,ch] - bg * 0.8, 0, None)
    mx = proc.max()
    if mx > 0: proc = proc / mx
    # Denoise
    if denoise > 0:
        p8 = (np.clip(proc, 0, 1) * 255).astype(np.uint8)
        proc = cv2.bilateralFilter(p8, denoise, 30, 30).astype(np.float32) / 255.0
    return proc

def update_postprocess(**kwargs):
    with out_pp:
        clear_output(wait=True)
        result = apply_postprocess(
            preview_pre,
            pp['R'].value, pp['G'].value, pp['B'].value,
            pp['sat'].value, pp['scnr'].value, pp['gamma'].value, pp['denoise'].value
        )
        fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
        axes[0].imshow(preview_pre, origin='lower')
        axes[0].set_title('Original', fontsize=11); axes[0].axis('off')
        axes[1].imshow(result, origin='lower')
        axes[1].set_title('Post-procesado', fontsize=11); axes[1].axis('off')
        plt.suptitle(f'{OBJECT} - {MODE} Post-procesamiento', fontsize=13)
        plt.tight_layout(); plt.show()

for ctrl in pp.values():
    ctrl.observe(lambda c: update_postprocess(), names='value')

col1 = widgets.VBox([
    widgets.HTML('<b>Balance de color</b>'),
    pp['R'], pp['G'], pp['B'],
])
col2 = widgets.VBox([
    widgets.HTML('<b>Ajustes globales</b>'),
    pp['sat'], pp['scnr'], pp['gamma'], pp['denoise'],
])

display(widgets.VBox([
    widgets.HTML(f'<h3>Post-procesamiento: {MODE} ({OBJECT})</h3>'),
    widgets.HBox([col1, widgets.HTML('&nbsp;&nbsp;&nbsp;'), col2]),
]))
display(out_pp)
update_postprocess()

## Paso 13: Aplicar post-procesado a resolución completa

Cuando estés conforme con el preview, ejecuta esta celda.
Aplica exactamente los mismos parámetros a la imagen de 61 MP.

In [ ]:
print("Aplicando post-procesado a resolución completa...\n")
print(f"  R={pp['R'].value}, G={pp['G'].value}, B={pp['B'].value}")
print(f"  Sat={pp['sat'].value}, SCNR={pp['scnr'].value}")
print(f"  Gamma={pp['gamma'].value}, Denoise={pp['denoise'].value}")

final_proc = apply_postprocess(
    final_pre,
    pp['R'].value, pp['G'].value, pp['B'].value,
    pp['sat'].value, pp['scnr'].value, pp['gamma'].value, pp['denoise'].value
)

print("\nPost-procesado aplicado a resolución completa.")

## Paso 14: Exportacion

- **PNG** (8-bit): para ver y compartir
- **TIFF** (16-bit): para post-procesamiento adicional en PixInsight o Photoshop

In [ ]:
obj_clean = OBJECT.replace(' ', '_')
prefix = f"{obj_clean}_{MODE}_final"

png_path = OUTPUT_DIR / f'{prefix}.png'
img8 = (np.clip(final_proc, 0, 1) * 255).astype(np.uint8)
Image.fromarray(np.flipud(img8)).save(str(png_path), 'PNG')
print(f"  PNG:  {png_path.name}  ({os.path.getsize(str(png_path))/1e6:.1f} MB)")

tiff_path = OUTPUT_DIR / f'{prefix}.tiff'
img16 = (np.clip(final_proc, 0, 1) * 65535).astype(np.uint16)
cv2.imwrite(str(tiff_path), np.flipud(img16[:,:,::-1]))
print(f"  TIFF: {tiff_path.name}  ({os.path.getsize(str(tiff_path))/1e6:.1f} MB)")

fig, ax = plt.subplots(figsize=(14, 9))
ax.imshow(np.flipud(img8))
ax.set_title(f'{OBJECT} - {MODE} Final', fontsize=14); ax.axis('off')
plt.tight_layout(); plt.show()

print(f"\nArchivos en: {OUTPUT_DIR.absolute()}")

## Referencia rápida

### Pipeline completo:

```
  FITS crudos
     |
  [1] Calibración: (light - (dark - pedestal)) / flat
     |
  [2] Alineación intra-filtro (phase correlation)
     |
  [3] Apilado (mediana)
     |
  [4] Alineación inter-filtro (sliders interactivos)
     |
  [5] Composición (SHO o RGB) + stretch
     |
  [6] LRGB (opcional)
     |
  [7] Post-procesado (sliders interactivos)
     |
  PNG + TIFF finales
```

### Parámetros por modo:

| Parámetro | SHO | RGB | HaRGB / HaOIII-RGB |
|-----------|-----|-----|---------------------|
| STRETCH | 0.05-0.1 | 0.2-0.5 | 0.2-0.5 |
| GAMMA | 3.0-4.0 | 2.0-3.0 | 2.0-3.0 |
| SAT_FACTOR | 1.5-2.5 | 1.0-1.5 | 1.2-1.8 |
| SCNR | 0.7-0.9 | >1.0 (off) | >1.0 (off) |
| G_FACTOR | 0.7-0.8 | 1.0 | 1.0 |
| NB_MIX | n/a | n/a | 0.2-0.5 |

### Problemas comunes:

| Problema | Solución |
|----------|----------|
| Imagen muy oscura | Bajar STRETCH, subir GAMMA o Gamma extra |
| Imagen muy verde | Bajar G en post-procesado, bajar SCNR |
| Imagen púrpura | Subir SCNR (a 0.95+), bajar B |
| Colores debiles/grises | Subir Saturación (siempre &ge; 1.0!) |
| Estrellas con franjas | Ajustar sliders de alineación |
| Mucho ruido | Subir Denoise (7-9) o capturar más frames |

### Para procesar otro objeto:

1. Vuelve al **Paso 4** y selecciona otro objeto en el dropdown
2. Selecciona el modo deseado (el dropdown se actualiza automáticamente)
3. Ejecuta todas las celdas de nuevo (Kernel &rarr; Restart & Run All)

### Para mejorar la captura:

- Más frames por filtro (10-20) mejora el SNR dramáticamente
- Mas exposición de luminancia (5-10 &times; 300s) para LRGB efectivo
- Dithering entre frames elimina patrones del sensor
- La calidad es proporcional al **tiempo total de integración**

### Si los sliders no aparecen:

```python
!pip install ipywidgets ipympl
!jupyter nbextension enable --py widgetsnbextension
```

Si usas JupyterLab: `!jupyter labextension install @jupyter-widgets/jupyterlab-manager`

Si `%matplotlib widget` da error, reemplázalo por `%matplotlib inline` en la celda de imports.